# ML-04 â€” Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/samarthjoshi02/flyrank-internship-ml/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** â€” each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

**Unit of analysis**: One row represents a single content item (`content_id`) for a specific pseudonymized `client_id`.

**Time window**: All metrics are pre-aggregated over a trailing 90-day window ending at export time. The comparison metrics (`last_30d`, `prev_30d`) are subsets of this 90-day window.


In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

data_path = Path('../../data/raw/content_refresh_anonymized.csv').resolve()
if not data_path.exists():
    data_path = Path('../data/raw/content_refresh_anonymized.csv').resolve()

df = pd.read_csv(data_path)

# Verify Unit of analysis (Grain)
grain_check = df.groupby('content_id').size()
print(f"Total rows: {len(df)}")
print(f"Unique content_ids: {len(grain_check)}")
print(f"Duplicate content_ids: {sum(grain_check > 1)}")


Total rows: 30000
Unique content_ids: 30000
Duplicate content_ids: 0


## 2. Fields: feature / label / context / excluded

*   **Feature**: Knowable before prediction. Examples: `search_volume`, `competition`, `cpc`, `word_count`, `char_count`, `impressions_90d`, `clicks_90d`, `pageviews_90d`, `content_age_days`, `impressions_prev_30d`, `ctr`, `avg_position`.
*   **Label**: `is_declining_label` (derived from `trend_direction`).
*   **Context**: `content_id`, `client_id`, `content_type`, `main_intent`. Grouping/stratification only.
*   **Excluded**: 
    *   `provider_used`, `model_used`: Not known before generation.
    *   `trend_pct`, `trend_direction`: Direct label inputs (target leakage).
    *   `impressions_last_30d`, `clicks_last_30d`: Overlap with the label's outcome window.


In [3]:
# Missingness follows content_type pattern (e.g., feedly articles lack keywords)
missing_pct = df.groupby('content_type')['search_volume'].apply(lambda x: x.isna().mean() * 100)
print("% Missing search_volume by content_type:")
print(missing_pct.round(2).astype(str) + '%')


% Missing search_volume by content_type:
content_type
comparison article      0.0%
feedly article        100.0%
keyword article        1.37%
Name: search_volume, dtype: object


## 3. Verify it with queries (grain, counts, missing values, windows)

1. **Grain**: `content_id` is unique per row.
2. **Counts**: Total rows match 30,000 as expected.
3. **Missingness**: Missing values follow structural patterns, not random.


In [4]:
# Query 1: Grain
duplicates = df.groupby('content_id').size().reset_index(name='c')
print("1. Duplicate content_ids:\n", duplicates[duplicates['c'] > 1])

# Query 2: Counts
client_counts = df.groupby('client_id').size()
print("\n2. Client counts summary:")
print(client_counts.describe())

# Query 3: Missing values
missing_summary = (df.isna().mean() * 100).sort_values(ascending=False)
print("\n3. Top 5 columns by missing percentage:")
print(missing_summary.head(5).round(2).astype(str) + '%')


1. Duplicate content_ids:
 Empty DataFrame
Columns: [content_id, c]
Index: []

2. Client counts summary:
count      32.000000
mean      937.500000
std      1376.387113
min         3.000000
25%       110.250000
50%       567.000000
75%      1058.750000
max      7008.000000
dtype: float64

3. Top 5 columns by missing percentage:
provider_used      71.46%
char_count         25.66%
word_count         25.66%
word_count_tier    25.66%
char_count_tier    25.66%
dtype: object


## 4. Data limits

*   **Missing GA4 Data**: Zeros in `ga4` columns before `ga4_data_start` are missing data, not zero engagement. Flags are three-valued (TRUE, FALSE, NULL).
*   **Window overlap**: The 90-day window for queries overlaps the recent 30-day window used for the label. Using full 90-day metrics can cause target leakage.
*   **Zero means missing for positions**: `avg_position = 0` means no position data, not rank 0.


In [5]:
# Check avg_position trap
zero_pos = len(df[df['avg_position'] == 0])
print(f"Rows with avg_position == 0 (no data): {zero_pos}")


Rows with avg_position == 0 (no data): 1205


## 5. Build the five-feature dataframe & Leakage Experiment

We build a dataframe with 5 safe features. We intentionally include `trend_pct` to demonstrate target leakage, show its correlation, and then remove it.


In [6]:
# Create label
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

# 1. Five safe features + label + leaked feature
safe_features = ['impressions_prev_30d', 'clicks_prev_30d', 'content_age_days', 'word_count', 'search_volume']
leaked_feature = 'trend_pct'

df_model = df[safe_features + [leaked_feature, 'is_declining_label']].copy()

# 2. Demonstrate leakage
print("Correlation with label (WITH LEAKAGE):")
print(df_model.corr(numeric_only=True)['is_declining_label'].sort_values(ascending=False))
print("\ntrend_pct has artificial massive negative correlation because the label is derived from it.")

# 3. Remove leakage
df_model = df_model.drop(columns=[leaked_feature])
print("\nFeatures after removing leakage:", df_model.columns.tolist())


Correlation with label (WITH LEAKAGE):
is_declining_label      1.000000
word_count              0.090157
impressions_prev_30d    0.004482
search_volume          -0.019103
clicks_prev_30d        -0.028716
trend_pct              -0.141068
content_age_days       -0.163882
Name: is_declining_label, dtype: float64

trend_pct has artificial massive negative correlation because the label is derived from it.

Features after removing leakage: ['impressions_prev_30d', 'clicks_prev_30d', 'content_age_days', 'word_count', 'search_volume', 'is_declining_label']
